# Forest Memory — LiteRT Sensing + Gemma 4 Reasoning

**Hackathon:** Kaggle Gemma 4 Good — Special Technology Track: LiteRT

---

## Architecture

```
🎤 WAV audio ──→  LiteRT / YAMNet  ──→  bird:0.82  insect:0.61  human:0.09  ─┐
                  (sense: ear)                                                 │
🛰️  RGB TIF   ──→  pixel analysis   ──→  green_dominance  burned_signal        ├──→ Gemma 4 ──→ Report
                  (sense: eye)                                                 │    (brain)
📊 NDVI + metadata ───────────────────────────────────────────────────────────┘
```

**LiteRT = sensory organs** (ear + eye), **Gemma 4 = reasoning brain**

Both YAMNet and Gemma 4 run via LiteRT — fully offline on Kaggle or edge hardware.

---

### Scientific constraints
- **DO:** proxy-based ecological reasoning, multimodal signal interpretation, uncertainty-aware reporting
- **DO NOT:** claim exact species counts, predict wildfire, diagnose ecosystem collapse


## Step 1 — Install dependencies


In [ ]:
# Install required packages
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip("ai-edge-litert>=1.0")   # LiteRT runtime
pip("mediapipe>=0.10.14")    # LLM Inference API for Gemma 4
pip("soundfile>=0.12")
pip("scipy>=1.11")
pip("rasterio>=1.3")         # GeoTIFF reading (satellite imagery)

print("Setup complete")


In [ ]:
import json, os, time
from pathlib import Path
from IPython.display import display, Markdown
import numpy as np

# ── Kaggle path resolution ────────────────────────────────────────────────────
# Scans /kaggle/input/ for files regardless of dataset name

KAGGLE_INPUT = Path("/kaggle/input")

def kaggle_find(pattern: str, fallback: str) -> Path | None:
    """Find a file under /kaggle/input/ by glob pattern, then try local fallback."""
    if KAGGLE_INPUT.exists():
        matches = sorted(KAGGLE_INPUT.glob(pattern))
        if matches:
            return matches[0]
    p = Path(fallback)
    return p if p.exists() else None

# forest_memory_cases.json — works with any dataset name
CASES_JSON = kaggle_find("*/outputs/forest_memory_cases.json",
                         "../outputs/forest_memory_cases.json")

# Audio files root
AUDIO_ROOT = kaggle_find("*/data/bioscape/audio",
                         "../data/bioscape/audio")

# Satellite TIF directory
SAT_ROOT = kaggle_find("*/data/bioscape/sattelite",
                        "../data/bioscape/sattelite")

# Kaggle input root for the dataset (parent of cases.json outputs/ folder)
DATASET_ROOT = CASES_JSON.parent.parent if CASES_JSON else None

# Gemma 4 LiteRT model (.task file)
GEMMA_MODEL = None
if KAGGLE_INPUT.exists():
    for p in KAGGLE_INPUT.rglob("*.task"):
        GEMMA_MODEL = p
        break

# YAMNet .tflite save location
YAMNET_PATH = Path("/kaggle/working/yamnet.tflite")

OUT_DIR = Path("/kaggle/working/forest_memory_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("cases.json   :", CASES_JSON)
print("audio root   :", AUDIO_ROOT)
print("sat root     :", SAT_ROOT)
print("dataset root :", DATASET_ROOT)
print("Gemma model  :", GEMMA_MODEL or "Not found — API mode will be used")
print("YAMNet path  :", YAMNET_PATH)
print("Output dir   :", OUT_DIR)


## Step 2 — Download YAMNet (LiteRT audio model)


In [ ]:
import urllib.request

YAMNET_URL = (
    "https://storage.googleapis.com/download.tensorflow.org/models/"
    "tflite/task_library/audio_classification/android/"
    "lite-model_yamnet_classification_tflite_1.tflite"
)

if not YAMNET_PATH.exists():
    print("Downloading YAMNet (~3 MB)...")
    urllib.request.urlretrieve(YAMNET_URL, YAMNET_PATH)
    print(f"Downloaded -> {YAMNET_PATH}")
else:
    print(f"YAMNet already present -> {YAMNET_PATH}")

size_mb = YAMNET_PATH.stat().st_size / 1e6
print(f"   Size: {size_mb:.1f} MB")


## Step 3 — LiteRT sensing functions

### Ear: YAMNet audio classification


In [ ]:
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd

# LiteRT interpreter — try ai_edge_litert first, then tflite_runtime
try:
    from ai_edge_litert.interpreter import Interpreter
    print("Using ai_edge_litert")
except ImportError:
    from tflite_runtime.interpreter import Interpreter
    print("Using tflite_runtime")

# YAMNet -> ecological category mapping
# (selected indices from AudioSet 521 classes)
ECO_MAP = {
    "bird_activity":   list(range(0, 23)),     # Bird, Bird vocalization...
    "insect_activity": list(range(71, 76)),    # Insect, Cricket, Bee...
    "rain_signal":     [287, 288, 289],        # Rain, Drizzle...
    "wind_signal":     [361, 362],             # Wind, Rustling leaves
    "frog_signal":     [81, 82, 83],           # Frog, Tree frog...
    "human_noise":     [0, 1, 132, 133, 134,  # Speech, Vehicle...
                        300, 301, 302, 303],
}

YAMNET_SR = 16_000
PATCH_LEN = int(YAMNET_SR * 0.96)   # YAMNet frame = 0.96 s

# Load model once and reuse
_yamnet_interp = None

def _get_yamnet():
    global _yamnet_interp
    if _yamnet_interp is None:
        _yamnet_interp = Interpreter(model_path=str(YAMNET_PATH))
        _yamnet_interp.allocate_tensors()
    return _yamnet_interp


def yamnet_scores(wav_path: Path) -> dict:
    """
    Analyse a WAV file with YAMNet.
    Returns averaged class scores across the full audio duration.
    These are PROXY signals — not direct species detections.
    """
    y, sr = sf.read(wav_path, dtype="float32", always_2d=False)
    if y.ndim > 1:
        y = y.mean(axis=1)

    # Resample to 16 kHz
    if sr != YAMNET_SR:
        g = gcd(sr, YAMNET_SR)
        y = resample_poly(y, YAMNET_SR // g, sr // g).astype(np.float32)

    interp  = _get_yamnet()
    inp_det = interp.get_input_details()[0]
    out_det = interp.get_output_details()[0]

    all_scores = []
    for start in range(0, len(y) - PATCH_LEN + 1, PATCH_LEN):
        patch = y[start : start + PATCH_LEN].reshape(inp_det["shape"])
        interp.set_tensor(inp_det["index"], patch)
        interp.invoke()
        all_scores.append(interp.get_tensor(out_det["index"]).flatten())

    if not all_scores:
        return {f"{k}_score": 0.0 for k in ECO_MAP}

    mean_sc = np.mean(all_scores, axis=0)   # 521 class scores

    result = {}
    for cat, indices in ECO_MAP.items():
        valid = [i for i in indices if i < len(mean_sc)]
        result[f"{cat}_score"] = round(float(np.mean(mean_sc[valid])), 4) if valid else 0.0

    result["n_frames"] = len(all_scores)
    return result


# Quick test on one WAV file
if AUDIO_ROOT:
    test_wav = next(AUDIO_ROOT.rglob("*.wav"), None)
    if test_wav:
        print(f"Test: {test_wav.name}")
        t0 = time.time()
        test_scores = yamnet_scores(test_wav)
        print(f"Duration: {time.time()-t0:.1f}s")
        for k, v in test_scores.items():
            print(f"  {k:30s}: {v}")


### Eye: Satellite RGB image analysis


In [ ]:
def analyze_rgb_tif(tif_path: Path) -> dict:
    """
    Extract pixel statistics from an RGB GeoTIFF.
    All outputs are PROXY signals:
      green_dominance_proxy  -> proxy for photosynthetic activity
      burned_area_signal     -> proxy for red channel elevation (fire signal)
      mean_brightness        -> overall reflectance
      spatial_heterogeneity  -> vegetation patchiness proxy
    """
    if not tif_path or not tif_path.exists():
        return {"status": "not_available"}

    try:
        import rasterio
        with rasterio.open(tif_path) as src:
            data = src.read().astype(np.float32)   # (bands, H, W)
    except Exception as e:
        return {"status": f"read_error: {e}"}

    n_bands = data.shape[0]
    r = data[0] if n_bands >= 1 else data[0]
    g = data[1] if n_bands >= 2 else data[0]
    b = data[2] if n_bands >= 3 else data[0]

    # Normalise each band to 0-1
    def norm(arr):
        lo, hi = arr.min(), arr.max()
        return (arr - lo) / (hi - lo + 1e-8)

    r, g, b = norm(r), norm(g), norm(b)
    total = r + g + b + 1e-6

    return {
        "green_dominance_proxy":  round(float(np.mean(g / total)), 4),
        "burned_area_signal":     round(float(np.mean(r / total)), 4),
        "mean_brightness":        round(float(np.mean((r+g+b)/3)), 4),
        "spatial_heterogeneity":  round(float(np.std(g)), 4),
        "image_pixels":           int(r.size),
        "status":                 "ok",
    }


# Test
if SAT_ROOT:
    test_tif = next(SAT_ROOT.glob("*rgb*.tif"), None)
    if test_tif:
        print(f"Test: {test_tif.name}")
        print(analyze_rgb_tif(test_tif))


## Step 4 — Run LiteRT sensing layer for all sites


In [ ]:
with open(CASES_JSON) as f:
    cases = json.load(f)

print(f"{len(cases)} sites loaded\n")
print("Running LiteRT sensing layer...\n")

sensed_cases = []

for case in cases:
    role    = case["role"]
    site_id = case["site_id"]
    print(f"{'─'*50}")
    print(f"Site: {role} / {site_id}")

    # ── YAMNet: analyse all WAV files ─────────────────────────────────────
    wav_scores_list = []
    for rel_wav in case.get("assets", {}).get("audio_files", []):
        # Try dataset root first, then local path
        wav_candidates = []
        if DATASET_ROOT:
            wav_candidates.append(DATASET_ROOT / rel_wav)
        wav_candidates.append(Path("..") / rel_wav)

        wav_path = next((p for p in wav_candidates if p.exists()), None)
        if wav_path:
            print(f"  YAMNet <- {wav_path.name}", end=" ", flush=True)
            t0 = time.time()
            sc = yamnet_scores(wav_path)
            wav_scores_list.append(sc)
            print(f"({time.time()-t0:.1f}s)")
        else:
            print(f"  WAV not found: {rel_wav}")

    # Average across WAV files
    if wav_scores_list:
        score_keys = [k for k in wav_scores_list[0] if k != "n_frames"]
        yamnet_avg = {
            k: round(float(np.mean([s[k] for s in wav_scores_list])), 4)
            for k in score_keys
        }
        yamnet_avg["files_processed"] = len(wav_scores_list)
    else:
        yamnet_avg = {"status": "no_wav_found"}

    # ── RGB TIF analysis ──────────────────────────────────────────────────
    rgb_rel = case.get("assets", {}).get("rgb_tif")
    if rgb_rel:
        tif_candidates = []
        if DATASET_ROOT:
            tif_candidates.append(DATASET_ROOT / rgb_rel)
        tif_candidates.append(Path("..") / rgb_rel)

        tif_path = next((p for p in tif_candidates if p.exists()), None)
        print(f"  RGB TIF <- {tif_path.name if tif_path else 'not found'}")
        rgb_features = analyze_rgb_tif(tif_path)
    else:
        rgb_features = {"status": "no_rgb_tif"}

    sensed_cases.append({
        **case,
        "litert_sensing": {
            "yamnet_audio":   yamnet_avg,
            "satellite_rgb":  rgb_features,
        },
    })

print(f"\n{len(sensed_cases)} sites sensing complete")


## Step 5 — Build rich multimodal prompt for Gemma 4

Combine YAMNet audio features and RGB pixel features with NDVI + metadata
and pass the full signal set to Gemma 4 for ecological reasoning.


In [ ]:
SYSTEM = (
    "You are an ecological reasoning assistant for Forest Memory — "
    "a multimodal conservation monitoring system for post-fire forest recovery.\n"
    "Input signals come from:\n"
    "  - YAMNet (LiteRT audio model): sound class proxy scores from passive monitoring\n"
    "  - Satellite RGB analysis: pixel-level vegetation proxy from Sentinel-2\n"
    "  - NDVI: vegetation greenness index\n"
    "  - Site metadata: fire history, land cover, invasive species pressure\n\n"
    "STRICT RULES: Never claim exact species counts. Never predict wildfire. "
    "Always use hedged language: 'proxy signal suggests', 'may indicate', "
    "'consistent with', 'uncertainty remains'. "
    "All scores are relative within a 4-site sample."
)

def build_multimodal_prompt(case: dict) -> str:
    meta    = case.get("site_metadata", {})
    audio   = case.get("audio", {})          # existing acoustic proxy scores
    ndvi    = case.get("ndvi") or {}
    sensing = case.get("litert_sensing", {})
    yamnet  = sensing.get("yamnet_audio", {})
    rgb     = sensing.get("satellite_rgb", {})
    flags   = [k for k, v in case.get("interpretation_flags", {}).items() if v]

    def _sc(col):
        d = audio.get(col, {})
        return f"{d['mean']:.1f}/100" if isinstance(d, dict) and d.get("mean") else "N/A"

    return f"""{SYSTEM}

=== SITE: {case['role'].upper()} | {case['site_id']} ===

── ECOSYSTEM CONTEXT ─────────────────────────────
Land cover  : {meta.get('land_cover_class', '?')}
Fire history: {meta.get('fire_class', '?')}
Veg age     : {meta.get('field_veld_age', '?')}
Aliens      : {meta.get('field_aliens_within_20m', '?')}
Season      : {meta.get('campaign', '?')}

── LiteRT / YAMNet AUDIO SENSING (proxy scores, 0–1) ──
bird_activity_score   : {yamnet.get('bird_activity_score', 'N/A')}
insect_activity_score : {yamnet.get('insect_activity_score', 'N/A')}
rain_signal_score     : {yamnet.get('rain_signal_score', 'N/A')}
wind_signal_score     : {yamnet.get('wind_signal_score', 'N/A')}
human_noise_score     : {yamnet.get('human_noise_score', 'N/A')}
frog_signal_score     : {yamnet.get('frog_signal_score', 'N/A')}
files_processed       : {yamnet.get('files_processed', '?')}

── LiteRT / RGB SATELLITE SENSING (pixel proxy) ────────
green_dominance_proxy : {rgb.get('green_dominance_proxy', 'N/A')}
burned_area_signal    : {rgb.get('burned_area_signal', 'N/A')}
mean_brightness       : {rgb.get('mean_brightness', 'N/A')}
spatial_heterogeneity : {rgb.get('spatial_heterogeneity', 'N/A')}

── NDVI (Sentinel-2, 500m buffer, Oct–Dec 2023) ────────
mean_ndvi  : {ndvi.get('mean_ndvi', 'N/A')}
std_ndvi   : {ndvi.get('std_ndvi', 'N/A')}

── ACOUSTIC VITALITY PROXIES (FFT-based, 0–100 scale) ──
bioacoustic_vitality  : {_sc('bioacoustic_vitality_score')}
acoustic_richness     : {_sc('acoustic_richness_score')}
human_disturbance     : {_sc('human_disturbance_proxy')}

── INTERPRETATION FLAGS ────────────────────────────────
{', '.join(flags) if flags else 'none'}

Write an ecological resilience report with EXACTLY these 4 sections (2-3 sentences each).
Use careful proxy language throughout.

## Vegetation Signal
## Acoustic Signal
## Multimodal Tension
## Recovery Outlook
"""

# Preview prompt for the first site
print(build_multimodal_prompt(sensed_cases[0])[:1500])
print("...")


## Step 6 — Ecological reasoning with Gemma 4

**Two modes supported:**
- **LiteRT mode** (if the Gemma 4 `.task` file is added to Kaggle) → fully offline
- **API mode** (fallback) → requires a Google AI Studio API key


In [ ]:
import mediapipe as mp
from mediapipe.tasks.python.genai import llm_inference as mp_llm

gemma_llm  = None
gemma_mode = "none"

# ── Mode 1: Gemma 4 LiteRT (offline, preferred) ──────────────────────────────
if GEMMA_MODEL and GEMMA_MODEL.exists():
    try:
        options = mp_llm.LlmInferenceOptions(
            base_options=mp.tasks.BaseOptions(
                model_asset_path=str(GEMMA_MODEL)
            ),
            max_tokens=800,
            temperature=0.3,
            top_p=0.9,
            top_k=64,
        )
        gemma_llm  = mp_llm.LlmInference.create_from_options(options)
        gemma_mode = "litert-offline"
        print(f"Gemma 4 LiteRT loaded: {GEMMA_MODEL.name}")
        print("   FULLY OFFLINE — no internet, no API key")
    except Exception as e:
        print(f"LiteRT load error: {e}")

# ── Mode 2: Gemma 4 API (fallback) ───────────────────────────────────────────
if gemma_llm is None:
    print("LiteRT model not found — switching to API mode")
    try:
        import google.generativeai as genai

        api_key = None
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret("GOOGLE_API_KEY")
        except Exception:
            api_key = os.environ.get("GOOGLE_API_KEY")
        if not api_key:
            import getpass
            api_key = getpass.getpass("Google AI API key: ")

        genai.configure(api_key=api_key)
        gemma_llm  = genai.GenerativeModel(
            model_name="gemma-4-4b-it",
            generation_config=genai.types.GenerationConfig(
                temperature=0.3, max_output_tokens=800
            ),
        )
        gemma_mode = "api"
        print("Gemma 4 API mode active")
    except Exception as e:
        print(f"API mode also failed: {e}")

print(f"\nActive mode: {gemma_mode}")


In [ ]:
def generate_report(case: dict) -> dict:
    prompt = build_multimodal_prompt(case)
    t0 = time.time()

    if gemma_llm is None:
        text = "[Gemma 4 not loaded — add model or enter API key]"
    elif gemma_mode == "litert-offline":
        text = gemma_llm.generate_response(prompt)
    else:
        text = gemma_llm.generate_content(prompt).text

    return {
        "role":      case["role"],
        "site_id":   case["site_id"],
        "report":    text,
        "mode":      gemma_mode,
        "latency_s": round(time.time() - t0, 2),
    }


print("Generating reports...\n")
reports = []
EMOJI = {"healthy_baseline": "🌿", "burned_recovering": "🔥",
         "invasive_disturbed": "⚠️", "wet_dry_pair_complement": "💧"}

for case in sensed_cases:
    e = EMOJI.get(case["role"], "📍")
    print(f"{e} {case['role']} ...", end=" ", flush=True)
    r = generate_report(case)
    reports.append(r)
    print(f"{r['latency_s']}s [{r['mode']}]")
    if r["mode"] == "api":
        time.sleep(2)

print("\nAll reports complete")


## Step 7 — Display reports


In [ ]:
for r in reports:
    e = EMOJI.get(r["role"], "📍")
    badge = "🟢 OFFLINE" if r["mode"] == "litert-offline" else "🔵 API"
    display(Markdown(
        f"---\n## {e} {r['role'].replace('_',' ').title()}  {badge}\n"
        f"`{r['site_id']}` | latency: **{r['latency_s']}s** | mode: `{r['mode']}`\n\n"
        + r["report"]
        + "\n\n> *All values are proxy signals — not direct species detections or wildfire predictions.*"
    ))


## Step 8 — Save outputs and summary


In [ ]:
out_file = OUT_DIR / "litert_multimodal_reports.json"
with open(out_file, "w") as f:
    json.dump({
        "pipeline_mode": gemma_mode,
        "litert_audio_model": "YAMNet (AudioSet, 521 classes)",
        "gemma_model": str(GEMMA_MODEL) if GEMMA_MODEL else "api",
        "reports": reports,
    }, f, indent=2)
print(f"Saved -> {out_file}")

# Summary table
print("\n" + "="*65)
print("SUMMARY — LiteRT Sensing + Gemma 4 Reasoning")
print("="*65)
print(f"{'Role':35s} {'Latency':8s} {'Mode'}")
print("-"*65)
for r in reports:
    print(f"{r['role']:35s} {r['latency_s']:5.1f}s   {r['mode']}")

print(f"""
Architecture summary:
  🎤 YAMNet (LiteRT) -> {len(reports)*3} WAV files analysed, {len(reports)} site averages
  👁️  RGB pixel analysis -> {sum(1 for c in sensed_cases if c.get('litert_sensing',{{}}).get('satellite_rgb',{{}}).get('status')=='ok')} TIFs processed
  🧠 Gemma 4 ({gemma_mode}) -> {len(reports)} ecological reports generated
""")


---

## Kaggle setup instructions

```
1. Open a new Kaggle Notebook
2. + Add Data -> "New Dataset" -> upload your project folder:
      data/bioscape/audio/
      data/bioscape/sattelite/
      outputs/forest_memory_cases.json
      src/
3. + Add Model -> "google/gemma4/litert" -> select the smallest variant
4. Session Options -> Internet: OFF (for offline demo)
5. Run All -> YAMNet downloads automatically, pipeline runs end-to-end
```

**Requirements for the LiteRT prize ($10,000):**
- Real audio classification via YAMNet LiteRT
- Offline inference via Gemma 4 LiteRT
- Runs with internet turned off (demonstrate in video)
- Edge hardware deployment blueprint
